# TeeUnit: Training LLMs to Play Teeworlds with GRPO

In this tutorial, we'll teach an LLM to play **Teeworlds** - a 2D multiplayer shooter game - using **reinforcement learning**. By the end, you'll understand how to:

- Connect LLMs to game environments using **OpenEnv**
- Design reward functions for combat games
- Train models with **GRPO** (Group Relative Policy Optimization)
- Upload trained models to **HuggingFace Hub**

**Requirements:** This notebook runs on a free Tesla T4 Google Colab instance.

## What is TeeUnit?

**TeeUnit** wraps the real Teeworlds 0.7.5 game as an OpenEnv-compatible environment for LLM training. The agent controls a "tee" (character) and must survive while eliminating opponents using various weapons (pistol, shotgun, grenade launcher, laser, and the iconic grappling hook).

## Our Goal

We'll use GRPO to train an LLM that receives game state as text and outputs actions like "move right", "jump", "aim at enemy", and "shoot". The model learns to:
- **Survive** longer in combat
- **Aim accurately** at enemies
- **Use weapons strategically**
- **Navigate** the arena effectively

## Installation

We need these key libraries:

1. **[OpenEnv](https://github.com/meta-pytorch/OpenEnv)** - Universal RL environment interface
2. **[Unsloth](https://github.com/unslothai/unsloth)** - Memory-efficient training (~70% VRAM reduction)
3. **[TRL](https://github.com/huggingface/trl)** - Transformer Reinforcement Learning library

In [ ]:
%%capture
import os, importlib.util

!pip install --upgrade -qqq uv

# Install PyTorch, Unsloth, and dependencies
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy
        get_numpy = f"numpy=={numpy.__version__}"
    except:
        get_numpy = "numpy"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth trackio

!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
%%capture
# Install OpenEnv and TeeUnit environment
!pip install -qqq openenv-core websockets fastmcp fastapi

# Clone TeeUnit repo for the environment
!git clone -q https://github.com/ziadgit/teeunit.git /content/teeunit 2>/dev/null || true

import sys
sys.path.insert(0, '/content/teeunit')

## Loading the Model

We use Qwen2.5-3B-Instruct with 4-bit quantization for memory efficiency.

| Parameter | Value | Description |
|-----------|-------|-------------|
| `max_seq_length` | 1024 | Maximum context length for game state + response |
| `load_in_4bit` | True | Quantizes weights to 4-bit for VRAM savings |
| `lora_rank` | 8 | LoRA adapter rank for efficient fine-tuning |

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024
lora_rank = 8

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=True,
    offload_embedding=True,  # Save ~1GB VRAM
)

print(f"Model loaded! Max sequence length: {max_seq_length}")

In [ ]:
# Add LoRA adapters for efficient training
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank * 2,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA adapters added!")

## Setting Up the TeeUnit Environment

We'll use the TeeUnit OpenEnv environment. For this demo, we use a local simulation.
In production, this connects to a real Teeworlds server.

In [ ]:
from teeunit_env.server.tee_environment import TeeEnvironment

# Create environment instance
env = TeeEnvironment(num_agents=4, max_steps=200)

# Reset and get initial state
obs = env.reset()
print("Environment ready!")
print(f"Episode ID: {obs.metadata['episode_id']}")
print("\nInitial game state:")
print(obs.metadata['message'])

In [ ]:
# Test the MCP tools
mcp = env._mcp

# Get current status
for tool in mcp._tools.values():
    if tool.name == "get_status":
        status = tool.fn()
        print(status)
        break

In [ ]:
# Test some actions
def call_tool(name, **kwargs):
    for tool in mcp._tools.values():
        if tool.name == name:
            return tool.fn(**kwargs)
    return f"Unknown tool: {name}"

print(call_tool("move", direction="right"))
print(call_tool("jump"))
print(call_tool("aim", x=500, y=300))
print(call_tool("shoot", weapon=2))
print()
print(call_tool("get_status"))

## Prompt Design

The LLM receives the game state as text and must output a valid action.
We structure the prompt to be clear and concise.

In [ ]:
SYSTEM_PROMPT = """You are an expert Teeworlds player. Based on the game state, output your next action.

Available actions:
- MOVE LEFT / MOVE RIGHT / STOP
- JUMP
- AIM x y (where x, y are coordinates)
- SHOOT [weapon] (weapon: 0=hammer, 1=pistol, 2=shotgun, 3=grenade, 4=laser)
- HOOK

Output ONE action per response. Think strategically:
- Aim at nearby enemies before shooting
- Use cover and movement to avoid damage
- Switch weapons based on distance (shotgun close, laser far)
- Use the hook to escape or chase enemies
"""

def format_prompt(game_state: str) -> str:
    """Format the prompt for the LLM."""
    return f"{game_state}\n\nWhat is your next action?"

# Test prompt
test_state = call_tool("get_status")
test_prompt = format_prompt(test_state)
print("Sample prompt:")
print("-" * 50)
print(test_prompt)

In [ ]:
import re

def parse_action(response: str) -> tuple[str, dict]:
    """
    Parse LLM response into action name and arguments.
    
    Returns:
        (tool_name, arguments)
    """
    response = response.strip().upper()
    
    # Move actions
    if "MOVE LEFT" in response or "LEFT" in response:
        return "move", {"direction": "left"}
    if "MOVE RIGHT" in response or "RIGHT" in response:
        return "move", {"direction": "right"}
    if "STOP" in response:
        return "move", {"direction": "none"}
    
    # Jump
    if "JUMP" in response:
        return "jump", {}
    
    # Hook
    if "HOOK" in response:
        return "hook", {}
    
    # Aim - extract coordinates
    aim_match = re.search(r'AIM\s+(\d+)\s+(\d+)', response)
    if aim_match:
        x, y = int(aim_match.group(1)), int(aim_match.group(2))
        return "aim", {"x": x, "y": y}
    
    # Shoot - extract weapon
    shoot_match = re.search(r'SHOOT\s*(\d)?', response)
    if shoot_match or "SHOOT" in response or "FIRE" in response:
        weapon = int(shoot_match.group(1)) if shoot_match and shoot_match.group(1) else -1
        return "shoot", {"weapon": weapon}
    
    # Default: get status (safe action)
    return "get_status", {}

# Test parser
test_responses = [
    "MOVE RIGHT",
    "I'll JUMP to dodge!",
    "AIM 500 300 and prepare to shoot",
    "SHOOT 2",
    "Let me HOOK to escape",
]

for resp in test_responses:
    action, args = parse_action(resp)
    print(f"{resp!r} -> {action}({args})")

## Reward Functions

GRPO uses reward functions to score model outputs. We define rewards for:

1. **Valid Action** - Did the model output a parseable action?
2. **Survival** - Is the player still alive?
3. **Combat** - Did the player deal damage or get kills?

In [ ]:
# Global environment for reward computation
# In real training, each rollout gets its own environment
_reward_env = None
_reward_step = 0

def get_reward_env():
    global _reward_env, _reward_step
    if _reward_env is None:
        _reward_env = TeeEnvironment(num_agents=4, max_steps=50)
        _reward_env.reset()
        _reward_step = 0
    return _reward_env

def reset_reward_env():
    global _reward_env, _reward_step
    if _reward_env is not None:
        _reward_env.reset()
        _reward_step = 0


def valid_action_reward(completions, **kwargs) -> list[float]:
    """
    Reward for outputting a valid, parseable action.
    
    +0.5 for valid action
    -0.5 for invalid/unparseable action
    """
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        action, args = parse_action(text)
        
        # Check if we got a real action (not just get_status default)
        if action != "get_status":
            rewards.append(0.5)
        else:
            # Check if get_status was explicitly requested
            if "STATUS" in text.upper() or "STATE" in text.upper():
                rewards.append(0.3)
            else:
                rewards.append(-0.5)  # Unparseable
    
    return rewards


def survival_reward(completions, **kwargs) -> list[float]:
    """
    Reward for staying alive after executing action.
    
    +0.2 for surviving
    -1.0 for dying
    """
    rewards = []
    env = get_reward_env()
    
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        action_name, args = parse_action(text)
        
        # Execute action in environment
        mcp = env._mcp
        for tool in mcp._tools.values():
            if tool.name == action_name:
                try:
                    tool.fn(**args)
                except:
                    pass
                break
        
        # Simulate tick
        env._simulate_tick()
        
        # Check if player alive
        player = env._agents.get(env._current_agent_id)
        if player and player.is_alive:
            rewards.append(0.2)
        else:
            rewards.append(-1.0)
            reset_reward_env()  # Reset for next completion
    
    return rewards


def combat_reward(completions, **kwargs) -> list[float]:
    """
    Reward for combat effectiveness.
    
    +2.0 for killing an enemy
    +0.5 for dealing damage (shooting successfully)
    +0.1 for aiming at an enemy
    """
    rewards = []
    env = get_reward_env()
    
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        action_name, args = parse_action(text)
        
        reward = 0.0
        prev_kills = len([e for e in env._kill_events if e["killer_id"] == env._current_agent_id])
        
        # Execute action
        mcp = env._mcp
        result = ""
        for tool in mcp._tools.values():
            if tool.name == action_name:
                try:
                    result = tool.fn(**args)
                except:
                    pass
                break
        
        # Check for kills
        new_kills = len([e for e in env._kill_events if e["killer_id"] == env._current_agent_id])
        if new_kills > prev_kills:
            reward += 2.0 * (new_kills - prev_kills)
        
        # Check for damage dealt (from result text)
        if "Hit" in str(result) or "damage" in str(result).lower():
            reward += 0.5
        
        # Check for aiming at enemy
        if action_name == "aim":
            reward += 0.1
        
        rewards.append(reward)
    
    return rewards


print("Reward functions defined!")

## Creating the Training Dataset

We create prompts with game states for the model to learn from.

In [ ]:
from datasets import Dataset

def generate_game_state() -> str:
    """Generate a random game state for training."""
    env = TeeEnvironment(num_agents=4, max_steps=50)
    env.reset()
    
    # Simulate a few random ticks
    import random
    for _ in range(random.randint(1, 10)):
        env._simulate_tick()
    
    # Get status
    mcp = env._mcp
    for tool in mcp._tools.values():
        if tool.name == "get_status":
            return tool.fn()
    return ""

# Generate training prompts
NUM_SAMPLES = 500  # Increase for better training

training_prompts = []
for i in range(NUM_SAMPLES):
    game_state = generate_game_state()
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": format_prompt(game_state)},
    ]
    training_prompts.append({"prompt": prompt, "answer": 0})
    
    if (i + 1) % 100 == 0:
        print(f"Generated {i + 1}/{NUM_SAMPLES} prompts")

dataset = Dataset.from_list(training_prompts)

# Calculate max prompt length
max_prompt_length = max(
    len(tokenizer.apply_chat_template(p["prompt"], add_generation_prompt=True))
    for p in training_prompts[:10]
) + 50  # Buffer

print(f"\nDataset created with {len(dataset)} samples")
print(f"Max prompt length: {max_prompt_length}")
print(f"\nSample prompt:")
print(dataset[0]["prompt"][1]["content"][:500])

## Training with GRPO

**Group Relative Policy Optimization (GRPO)** computes advantages by comparing generations within the same prompt group.

Key parameters:
- `num_generations=2`: Generate 2 candidates per prompt
- `max_steps=200`: Total training steps (~1-2 hours on T4)
- `temperature=1.0`: Higher = more exploration

In [ ]:
from trl import GRPOConfig, GRPOTrainer

max_completion_length = max_seq_length - max_prompt_length

training_args = GRPOConfig(
    temperature=1.0,
    learning_rate=2e-4,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_generations=2,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=200,  # Increase for better results (600+ recommended)
    save_steps=50,
    report_to="none",  # Use "wandb" or "trackio" for logging
    output_dir="outputs",
)

print(f"Training config:")
print(f"  Max prompt length: {max_prompt_length}")
print(f"  Max completion length: {max_completion_length}")
print(f"  Total steps: {training_args.max_steps}")

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        valid_action_reward,
        survival_reward,
        combat_reward,
    ],
    args=training_args,
    train_dataset=dataset,
)

print("Trainer ready!")

In [ ]:
print("Starting GRPO training...")
print("Watch the reward column increase over time!")
print("="*60)

trainer.train()

print("\n" + "="*60)
print("Training complete!")

## Evaluating the Trained Model

Let's test how well the model plays Teeworlds now!

In [ ]:
def play_episode(model, tokenizer, max_steps=20):
    """Play one episode with the trained model."""
    env = TeeEnvironment(num_agents=4, max_steps=max_steps)
    env.reset()
    
    mcp = env._mcp
    
    def get_status():
        for tool in mcp._tools.values():
            if tool.name == "get_status":
                return tool.fn()
        return ""
    
    def execute_action(name, **kwargs):
        for tool in mcp._tools.values():
            if tool.name == name:
                return tool.fn(**kwargs)
        return "Unknown action"
    
    total_reward = 0
    kills = 0
    
    for step in range(max_steps):
        # Get game state
        state = get_status()
        
        # Format prompt
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": format_prompt(state)},
        ]
        
        # Generate action
        inputs = tokenizer.apply_chat_template(
            messages, 
            add_generation_prompt=True, 
            return_tensors="pt"
        ).to(model.device)
        
        outputs = model.generate(
            inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
        
        response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        
        # Parse and execute action
        action_name, args = parse_action(response)
        prev_kills = env._agents[0].score if env._agents.get(0) else 0
        
        result = execute_action(action_name, **args)
        env._simulate_tick()
        
        # Track rewards
        new_kills = env._agents[0].score if env._agents.get(0) else 0
        if new_kills > prev_kills:
            kills += (new_kills - prev_kills)
            total_reward += 2.0
        
        player = env._agents.get(0)
        if player and player.is_alive:
            total_reward += 0.1
        else:
            print(f"Step {step}: DIED")
            break
        
        if step % 5 == 0:
            print(f"Step {step}: {action_name}({args}) -> {result[:50]}...")
    
    return total_reward, kills, step + 1

# Play a few episodes
print("Playing evaluation episodes...\n")

total_rewards = []
total_kills = []
total_steps = []

for ep in range(3):
    print(f"\n--- Episode {ep + 1} ---")
    reward, kills, steps = play_episode(model, tokenizer, max_steps=20)
    total_rewards.append(reward)
    total_kills.append(kills)
    total_steps.append(steps)
    print(f"Episode {ep + 1}: Reward={reward:.2f}, Kills={kills}, Steps={steps}")

print(f"\n=== Evaluation Summary ===")
print(f"Average reward: {sum(total_rewards)/len(total_rewards):.2f}")
print(f"Average kills: {sum(total_kills)/len(total_kills):.2f}")
print(f"Average steps: {sum(total_steps)/len(total_steps):.2f}")

## Saving and Uploading to HuggingFace Hub

Let's save the trained model and upload it to HuggingFace!

In [ ]:
# Save locally first
model.save_pretrained("teeunit-agent-lora")
tokenizer.save_pretrained("teeunit-agent-lora")

print("Model saved locally to 'teeunit-agent-lora'")

In [ ]:
# Upload to HuggingFace Hub
# Set your HF token (get yours from https://huggingface.co/settings/tokens)
HF_TOKEN = "your_hf_token_here"  # Replace with your HF token
HF_REPO = "your-username/teeunit-agent"  # Replace with your HF repo

from huggingface_hub import login
login(token=HF_TOKEN)

# Push to hub
model.push_to_hub(
    HF_REPO,
    token=HF_TOKEN,
    commit_message="TeeUnit GRPO-trained agent (Qwen2.5-3B + LoRA)",
)
tokenizer.push_to_hub(
    HF_REPO,
    token=HF_TOKEN,
)

print(f"\nModel uploaded to: https://huggingface.co/{HF_REPO}")

## Conclusion

You've successfully:

1. Connected an LLM to the **TeeUnit** Teeworlds environment using **OpenEnv**
2. Designed **reward functions** for combat gameplay
3. Trained the model with **GRPO** using **Unsloth** for memory efficiency
4. Uploaded the trained model to **HuggingFace Hub**

### Next Steps

- **More training steps**: Increase `max_steps` to 600+ for better results
- **Larger models**: Try Qwen2.5-7B or 14B for more capable agents
- **Real server**: Connect to a real Teeworlds server for multiplayer training
- **Multi-agent**: Train agents to compete against each other (self-play)

### Resources

- [OpenEnv GitHub](https://github.com/meta-pytorch/OpenEnv)
- [Unsloth GitHub](https://github.com/unslothai/unsloth)
- [TRL Documentation](https://huggingface.co/docs/trl)
- [GRPO Paper](https://arxiv.org/abs/2402.03300)
- [TeeUnit GitHub](https://github.com/ziadgit/teeunit)